In [1]:
import torch
import random
import os
import numpy as np
# print(torch.cuda.is_available())  # True if GPU is available
# print(torch.cuda.device_count())  # Number of GPUs
# print(torch.cuda.get_device_name(0))  # GPU model name

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 99
seed_everything(seed);

from google.colab import drive
import sys
drive.mount('/content/drive')
os.chdir("drive/My Drive/Colab Notebooks")
sys.path.append('/content/drive/My Drive/Colab Notebooks/uEDA-Net/')

Mounted at /content/drive


In [2]:
!pip install torchinfo colorednoise

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from data_preprocess import subject_split_and_stack

folder_path = os.path.join(os.getcwd(), 'uEDA-Net')
data_path = os.path.join(folder_path, 'data')

noisy_train, noisy_test, target_train, target_test, noise_bank, _ = subject_split_and_stack(dataset_path=data_path,test_ratio=0.2,seed=seed, load_all = True)

📥 Loading dataset: /content/drive/My Drive/Colab Notebooks/uEDA-Net/data/processed_datasets.pt
✔ All datasets loaded successfully!
📥 Loading dataset: /content/drive/My Drive/Colab Notebooks/uEDA-Net/data/validation_set.pt

DATASET SPLIT SUMMARY
----------------------------------
Train Noisy : (5643, 512)
Train Clean : (5643, 512)
----------------------------------
Test Noisy  : (1547, 512)
Test Clean  : (1547, 512)
----------------------------------


In [4]:
def batch_corrcoef(y_hat, y, eps=1e-8):
    if y_hat.ndim == 3:
        y_hat = y_hat.squeeze(1)
        y = y.squeeze(1)

    y_hat_mean = y_hat.mean(dim=-1, keepdim=True)
    y_mean = y.mean(dim=-1, keepdim=True)
    y_hat_centered = y_hat - y_hat_mean
    y_centered = y - y_mean

    cov = torch.sum(y_hat_centered * y_centered, dim=-1)
    std_y_hat = torch.sqrt(torch.sum(y_hat_centered ** 2, dim=-1) + eps)
    std_y = torch.sqrt(torch.sum(y_centered ** 2, dim=-1) + eps)

    corr = cov / (std_y_hat * std_y + eps)
    return corr.mean()

def snr_db(noisy, output, clean, eps=1e-8):
    if clean.ndim == 3:
        clean = clean.squeeze(1)
        output = output.squeeze(1)
        noisy = noisy.squeeze(1)

    noise_in_power = torch.sum((noisy - clean) ** 2, dim=-1)
    noise_out_power = torch.sum((output - clean) ** 2, dim=-1)
    ratio = (noise_in_power + eps) / (noise_out_power + eps)
    snr = 10 * torch.log10(ratio)
    return snr.mean().item()

In [5]:
X_train = torch.tensor(noisy_train, dtype=torch.float32).unsqueeze(1)
y_train = torch.tensor(target_train, dtype=torch.float32).unsqueeze(1)
X_test  = torch.tensor(noisy_test, dtype=torch.float32).unsqueeze(1)
y_test  = torch.tensor(target_test, dtype=torch.float32).unsqueeze(1)

In [14]:
from model.unet_teacher import Denoising_Youngbin
from model.unet_student import Student_Youngbin
from model.Billal_UNet import denoising_model
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student_model = Denoising_Youngbin().to(device) # Student_Youngbin Denoising_Youngbin denoising_model
student_model.load_state_dict(torch.load("uEDA-Net/weights/teacher_model_best.pth", map_location=device))
model = student_model
model.eval()

mae_list, rmse_list, mse_list = [], [], []
snr_list, corr_list = [], []

with torch.no_grad():
    for i in range(X_test.shape[0]):
        noisy_input = X_test[i:i+1].to(device)
        clean_target = y_test[i:i+1].to(device)

        mean = noisy_input.mean(dim=2, keepdim=True)
        std = noisy_input.std(dim=2, keepdim=True).clamp(min=1e-6)
        noisy_input_norm = (noisy_input - mean) / std
        stats_flat = torch.cat([mean.view(-1, 1), std.view(-1, 1)], dim=1)

        denoised_out_norm = model(noisy_input_norm, stats_flat)
        denoised_out = denoised_out_norm * std + mean

        # losses
        loss_mse = F.mse_loss(denoised_out, clean_target)
        loss_mae = F.l1_loss(denoised_out, clean_target)
        loss_rmse = torch.sqrt(loss_mse)

        mse_list.append(loss_mse.item())
        mae_list.append(loss_mae.item())
        rmse_list.append(loss_rmse.item())

        # SNR & PCC
        snr_list.append(snr_db(noisy_input, denoised_out, clean_target))
        corr_list.append(batch_corrcoef(denoised_out, clean_target))

# mean ± std
def mean_std(x):
    return np.mean(x), np.std(x)

mae_m, mae_s = mean_std(mae_list)
rmse_m, rmse_s = mean_std(rmse_list)
mse_m, mse_s = mean_std(mse_list)
snr_m, snr_s = mean_std(snr_list)
corr_m, corr_s = mean_std(corr_list)

# print (.3f)
print(f"MAE: {mae_m:.3f} ± {mae_s:.3f} | RMSE: {rmse_m:.3f} ± {rmse_s:.3f}")
print(f"PCC: {corr_m:.3f} ± {corr_s:.3f} | SNR Imp: {snr_m:.3f} ± {snr_s:.3f} dB")


MAE: 0.125 ± 0.187 | RMSE: 0.160 ± 0.227
PCC: 0.909 ± 0.169 | SNR Imp: 13.170 ± 6.744 dB


In [15]:
import matplotlib.pyplot as plt
model.eval()
idxs = np.random.randint(1,1547,size = 20)

noisy_list = []
denoised_list = []
for idx in idxs:
    with torch.no_grad():
        noisy_input  = X_test[idx:idx+1].to(device)
        clean_target = y_test[idx:idx+1].to(device)
        mean = noisy_input.mean(dim=2, keepdim=True)
        std  = noisy_input.std(dim=2, keepdim=True).clamp(min=1e-3)
        stats_flat = torch.cat([mean.view(-1, 1), std.view(-1, 1)], dim=1)
        noisy_input_norm = (noisy_input - mean) / std
        denoised_out_norm = model(noisy_input_norm,stats_flat)
        denoised_out = denoised_out_norm*std + mean

    noisy_np    = noisy_input.squeeze().cpu().numpy()
    clean_np    = clean_target.squeeze().cpu().numpy()
    denoised_np = denoised_out.squeeze().cpu().numpy()

    plt.figure(figsize=(12, 8))
    plt.subplot(3, 1, 1)
    plt.plot(noisy_np, color='tab:orange')
    plt.title(f"Noisy Input {idx}")
    plt.ylabel("Amplitude")
    plt.grid(alpha=0.3)

    plt.subplot(3, 1, 2)
    plt.plot(denoised_np, color='tab:blue')
    plt.title(f"Denoised Output (Model Prediction)")
    plt.ylabel("Amplitude")
    plt.grid(alpha=0.3)

    plt.subplot(3, 1, 3)
    plt.plot(clean_np, color='tab:green')
    plt.title("Clean Target (Reference)")
    plt.xlabel("Time (samples)")
    plt.ylabel("Amplitude")
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


Output hidden; open in https://colab.research.google.com to view.

In [16]:
!pip install thop

In [22]:
from model.unet_teacher import Denoising_Youngbin
from model.unet_student import Student_Youngbin
from model.Billal_UNet import denoising_model
import torch.nn.functional as F
import torch
import time
import os
from thop import profile


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student_model = Denoising_Youngbin().to(device) # Student_Youngbin Denoising_Youngbin denoising_model
student_model.load_state_dict(torch.load("uEDA-Net/weights/teacher_model_best.pth", map_location=device))
model = student_model
model.eval()

dummy_input = torch.randn(1, 1, 512)
dummy_input2 = torch.randn(1, 2)
total_params = sum(p.numel() for p in model.parameters())

model_size_mb = (total_params * 4) / (1024 ** 2)

print(f"--- Complexity Metrics ---")
print(f"Total Parameters: {total_params / 1e6:.6f} M")
print(f"Estimated Model Size: {model_size_mb:.4f} MB")

macs, params = profile(model, inputs=(dummy_input,dummy_input2), verbose=False)
flops = macs * 2

print(f"--- Computation Metrics ---")
print(f"MACs: {macs / 1e6:.3f} M")
print(f"FLOPs: {flops / 1e6:.3f} MFLOPs\n")

device = torch.device('cpu')
model.to(device)
dummy_input = dummy_input.to(device)
dummy_input2 = dummy_input2.to(device)

print("Warming up CPU...")
with torch.no_grad():
    for _ in range(100):
        _ = model(dummy_input, dummy_input2)

num_iterations = 1000
total_time = 0.0

print(f"Running {num_iterations} iterations for latency measurement...")
with torch.no_grad():
    start_time = time.perf_counter()
    for _ in range(num_iterations):
        _ = model(dummy_input, dummy_input2)
    end_time = time.perf_counter()

total_time = end_time - start_time
average_latency_ms = (total_time / num_iterations) * 1000

print(f"--- Latency Metrics ---")
print(f"Average CPU Inference Latency: {average_latency_ms:.3f} ms")

--- Complexity Metrics ---
Total Parameters: 2.063041 M
Estimated Model Size: 7.8699 MB
--- Computation Metrics ---
MACs: 52.549 M
FLOPs: 105.097 MFLOPs

Warming up CPU...
Running 1000 iterations for latency measurement...
--- Latency Metrics ---
Average CPU Inference Latency: 6.496 ms
